# T1ME50 — High-Frequency Financial Forecasting

Fine-tunes the **T1ME50** (LLM for Time Series) model on 1-minute OHLCV data from the Nifty 50 index.

**Architecture**: GPT-2 backbone + Financial Patch Embedding (Conv1D FFT-based low-pass filter)

**Prediction Window**: 360 minutes (9:15 AM → 3:15 PM)

> **Runtime**: Set to **GPU (T4)** via `Runtime > Change runtime type > T4 GPU`

---
## Step 1 — Upload & Extract Project

In [ ]:
# Upload T1ME50_source.zip from your local machine
from google.colab import files
uploaded = files.upload()  # Select T1ME50_source.zip


In [ ]:
import zipfile, os

zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/T1ME50')

os.chdir('/content/T1ME50')
print('Working directory:', os.getcwd())
!ls -la


---
## Step 2 — Install Dependencies

In [ ]:
!pip install -q transformers==4.36.2 einops thop torchinfo sktime patool scikit-learn accelerate==0.26.0
print('\n✅ Dependencies installed.')


---
## Step 3 — Verify GPU

In [ ]:
import torch
print(f'PyTorch:  {torch.__version__}')
print(f'CUDA:     {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:      {torch.cuda.get_device_name(0)}')
    print(f'VRAM:     {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
else:
    print('⚠️  No GPU detected! Go to Runtime > Change runtime type > T4 GPU')


---
## Step 4 — Fetch Financial Data

In [ ]:
# Fetches 1-min OHLCV data for Nifty50 stocks via the Fyers / yfinance data procurement script
!python data_procurement/fetch_nifty50.py
!ls -lh data/finance/


---
## Step 5 — Train & Evaluate (360-Min Window, Top 10 Stocks)

In [ ]:
import os
import glob

os.makedirs('./results/graphs', exist_ok=True)

# Limit to first 10 stocks alphabetically
csv_files = sorted(glob.glob('./data/finance/*.csv'))[:10]
print(f'Running T1ME50 on {len(csv_files)} stocks with 360-min window\n')

for file_path in csv_files:
    stock_name = os.path.basename(file_path).replace('.csv', '')
    print(f"\n{'='*55}")
    print(f"  Training: {stock_name}")
    print(f"{'='*55}\n")

    # ── Train
    !python run_LLM4TS.py \
      --is_training 1 --is_llm 1 \
      --root_path ./data/finance/ \
      --data_path {stock_name}.csv \
      --model_id T1ME50_{stock_name}_1min \
      --model LLM4TS_pt --data finance \
      --features S --target Close \
      --seq_len 360 --label_len 120 --pred_len 360 \
      --patch_len 16 --stride 8 \
      --train_epochs 1 --batch_size 32 \
      --learning_rate 0.0001 \
      --freeze 0 --llm gpt2 \
      --use_gpu 1 --gpu 0 --itr 1

    # ── Evaluate & export JSON + CSV
    setting = f'T1ME50_{stock_name}_1min_sl360_pl360_llml1_lr0.0001_bs32_percent100_test_0'
    !python evaluate_financial_model.py \
      --setting "{setting}" \
      --data_path "{file_path}" \
      --stock_name "{stock_name}"

print('\n✅ All done. JSON files written to ./frontend/public/data/')
print('✅ Metrics appended to ./results/best_stocks_predictions.csv')


---
## Step 6 — Visualise Graphs

In [ ]:
from IPython.display import Image, display
import glob

graphs = sorted(glob.glob('./results/graphs/*.png'))
print(f'Found {len(graphs)} graphs')
for graph in graphs:
    print(graph)
    display(Image(filename=graph, width=900))


---
## Step 7 — Download Results

In [ ]:
import os, shutil, zipfile
from google.colab import files

# 1. Zip the frontend data folder (JSON payloads for the dashboard)
shutil.make_archive('frontend_data', 'zip', './frontend/public/data')
print('✅ frontend_data.zip created')

# 2. Zip the graphs
shutil.make_archive('forecast_graphs', 'zip', './results/graphs')
print('✅ forecast_graphs.zip created')

# 3. Download all three artefacts
for f in ['frontend_data.zip', 'forecast_graphs.zip', './results/best_stocks_predictions.csv', 'result.txt']:
    if os.path.exists(f):
        files.download(f)
        print(f'  ⬇  {f}')
    else:
        print(f'  ⚠  Not found: {f}')
